# Deep Learning para detección automática de COVID

En este proyecto, utilizaremos el dataset [COVID-19 Radiography Database](https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database/data) de Kaggle, es cual nos proporciona imágenes obtenidas mediante radiografías pertenecientes a tres tipos de pacientes diferentes: pacientes de control, sin ningún tipo de patoligía, pacientes de COVID-19 y pacientes con neumonía vírica.

*Nota: notebook adaptado y traducido del usuario de Kaggle [Ahmed Hafez](https://www.kaggle.com/code/ahmedtronic/covid-19-radiology-vgg19-f1-score-95)*

In [ ]:
# Nota: no hay por qué utilizar las librerías aquí puestas. Se pueden añadir/quitar
from IPython import display
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random as python_random
import tensorflow as tf
import seaborn as sns
import math
from sklearn.metrics import classification_report, confusion_matrix #, plot_confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)

La siguiente celda es para poder leer los datos directamente desde Google Colab. Si el alumno utiliza otra plataforma, se deberá saltar este paso y adaptar los PATHs a las carpetas de train y test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


En la siguiente celda, se define el PATH a nuestros datos. Aquí debemos adaptarlo en función dónde lo tengamos guardado. Si tenemos nuestra carpeta con los materiales *Máster DL - IEP* en la página de inicio de nuestro Drive, el siguiente PATH debería valernos.

*Nota: si no se utiliza Google Colab, se pueden utilizar paths relativos.*

In [ ]:
PATH = "drive/MyDrive/Máster DL - IEP/PROYECTO FINAL/"

Tras haber importado las librerías necesarias, vamos a generar un código que nos permita preprocesar las imágenes. Para ello, creamos la función *generate_data()*, a la cual le pasaremos el directorio donde tenemos almacenadas las imágenes.

Aquí encontramos la clase *ImageDataGenerator*, que ya hemos utilizado en notebooks anteriores. Veamos paso a paso lo que hacemos dentro de la función. Lo primero, es crear el objeto de la clase *ImageDataGenerator*, con la línea

```
datagen = ImageDataGenerator(rescale=1./255.)
```

donde lo único que les estamos indicando es que nos reescale las imágenes, como ya hemos hecho durante el curso, haciendo que el valor de cada píxel vaya de 0 a 1 en lugar de ir desde 0 a 255. Aquí es donde debemos darnos cuenta de que no estamos llevando a cabo ningún tipo de *Data Augmentation*. Cuando queremos generar el flujo de imágenes para aplicar el generador a nuestras imágenes, usamos

```
generator = datagen.flow_from_directory(DIR,
                                        batch_size=batch_size,
                                        shuffle=True,
                                        seed=42,
                                        class_mode='binary',
                                        target_size=(height, width),
                                        classes={'Normal': 0, 'Viral Pneumonia': 1,'Covid': 2})
```

Donde los parámetros más importantes para nosotros son:

- DIR es la carpeta donde están nuestras imágenes, a las cuales se aplicará el generador que hemos creado.
- batch_size es el tamaño del batch, que hemos definido anteriormente como 64.
- shuffle=True nos indica que el orden de los datos será aleatorio.
- target_size es el tamaño al que queremos reescalar las imágenes para adaptarlo a nuestro modelo, en nuestro caso 224x244 píxeles.
- classes es un diccionario para asignar a cada clase una etiqueta numérica. Así, posteriormente podremos entrenar nuestra red neuronal.

Tras ello, aplicamos el generador a nuestros conjuntos de entrenamiento y de test, obteniendo *test_generator* y *train_generator*.


In [ ]:
# ImageDataGenerator
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
from PIL import ImageFilter

height, width = 224, 224
batch_size=64


def generate_data(DIR):
    datagen = ImageDataGenerator(rescale=1./255.)

    generator = datagen.flow_from_directory(
        DIR,
        batch_size=batch_size,
        shuffle=True,
        seed=42,
        class_mode='binary',
        target_size=(height, width),
        classes={'Normal': 0, 'Viral Pneumonia': 1,'Covid': 2}
    )
    return generator

TRAINING_DIR = PATH + 'Covid19-dataset/train'
TESTING_DIR = PATH + 'Covid19-dataset/test'

train_generator = generate_data(TRAINING_DIR)
test_generator = generate_data(TESTING_DIR)

total_image = np.concatenate([train_generator.labels,test_generator.labels])

print('\n\n',{'Normal_cases':len(np.where(total_image==0)[0]),
      'Viral_Pneumonia_cases':len(np.where(total_image==1)[0]),
             'Covid_cases':len(np.where(total_image==2)[0])})

Found 0 images belonging to 3 classes.
Found 0 images belonging to 3 classes.


 {'Normal_cases': 0, 'Viral_Pneumonia_cases': 0, 'Covid_cases': 0}


Una vez tengamos los generadores creados, procedemos a ver cuántas muestras tenemos de cada clase obteniendo 90 de la clase 'Normal', otros 90 de la clase 'Neumonía vírica' y 137 de la clase 'COVID'.

## Desarrollo del proyecto

A partir de aquí, todo queda a criterio del alumno. Se pueden desarrollar modelos con cualquiera de las herramientas vistas a lo largo del curso - regularización, técnicas de entrenamiento, transfer learning, data augmentation, etc.

*Nota: el código anterior también puede ser modificado si el alumno lo desea.*